# Simplified knee output report

This notebook keeps only the logic needed to compute the knee-point output table for all 224 cities and save it as a CSV file.

In [ ]:
root_name = '251026'
root_dir = rf'../../data/output/mosa/{root_name}'
cs_gdf_dir = r'../../data/input/cs_gdf'
output_csv = rf'../../data/224cities_output.csv'

root_name, output_csv

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import CRS

from analysis import basic_stat, find_knee

warnings.simplefilter('ignore')
os.environ['GDAL_DATA'] = r'/usr/share/gdal'
pd.options.display.float_format = '{:.4f}'.format

def clearing(df):
    df = df.copy()
    df.columns = ['solution_no', 'obj1', 'obj2']
    df_numeric = df.select_dtypes(include=['number']).abs()
    df['obj1'] = df_numeric['obj1']
    df['obj2'] = df_numeric['obj2']

    rows_to_remove = []
    for index, row in df.iterrows():
        if any((df[['obj1', 'obj2']] < row.iloc[-2:]).all(axis=1)):
            rows_to_remove.append(index)

    return df.drop(rows_to_remove).copy()

In [ ]:
knee_rows = []
missing_items = []

for name in sorted(os.listdir(root_dir)):
    full_path = os.path.join(root_dir, name)
    if not (os.path.isdir(full_path) and name.endswith(f'_{root_name}')):
        continue

    city_name = name[:-len(f'_{root_name}')]
    print(f'Processing {city_name}...')

    cs_path = os.path.join(cs_gdf_dir, f'{city_name}.shp')
    obj_path = os.path.join(full_path, 'inf_archive_objs.csv')
    var_path = os.path.join(full_path, 'inf_archive_vars.csv')
    cv_path = os.path.join(full_path, 'inf_archive_cvs.csv')

    required_paths = [cs_path, obj_path, var_path, cv_path]
    missing = [p for p in required_paths if not os.path.exists(p)]
    if missing:
        missing_items.append({'city_name': city_name, 'missing_paths': ' | '.join(missing)})
        print(f'  Skipped: missing {len(missing)} required file(s)')
        continue

    cs_gdf = gpd.read_file(cs_path, crs=CRS.from_epsg(4547))
    cs_gdf = cs_gdf.drop_duplicates(subset=['node_id'], keep='first').reset_index(drop=True)
    cs_num = len(cs_gdf)

    obj_df = pd.read_csv(obj_path)
    var_df = pd.read_csv(var_path)
    cv_df = pd.read_csv(cv_path)

    obj_df = clearing(obj_df)
    stat_df = basic_stat(obj_df, var_df, cv_df, cs_num)
    knee_no = find_knee(stat_df)

    row = stat_df.loc[stat_df['solution_no'] == knee_no].copy()
    row['city_name'] = city_name
    row['candidate_num'] = cs_num
    row['built_ratio'] = row['bulit_num'] / row['candidate_num']
    knee_rows.append(row)

knee_df = pd.concat(knee_rows, ignore_index=True) if knee_rows else pd.DataFrame()
if not knee_df.empty:
    front = ['city_name']
    rest = [c for c in knee_df.columns if c not in front]
    knee_df = knee_df[front + rest].sort_values('candidate_num', ascending=False).reset_index(drop=True)

missing_df = pd.DataFrame(missing_items)

print(f'Finished. Collected {len(knee_df)} city rows.')
if len(missing_df):
    print(f'Missing entries: {len(missing_df)}')

knee_df.head()

In [ ]:
knee_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

if len(missing_df):
    missing_csv = output_csv.replace('.csv', '_missing.csv')
    missing_df.to_csv(missing_csv, index=False, encoding='utf-8-sig')
    print(f'Knee output saved to: {output_csv}')
    print(f'Missing-file report saved to: {missing_csv}')
else:
    print(f'Knee output saved to: {output_csv}')

knee_df